In [31]:
import numpy as np
import pandas as pd
import statistics

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)


In [32]:
data = pd.read_csv("../data/data.csv", index_col=0)
print(data)

      country time_of_day        lat       long       road_type  \
8459       SE         day  55.604209  13.028574            city   
63960      DE         day  50.936889   6.908079  arterial-urban   
71801      IT         day  41.987014  12.496538         highway   
85589      IT       night  41.794853  12.522880         highway   
90649      DE         day  49.182153   9.414737         highway   
...       ...         ...        ...        ...             ...   
74962      PL         day  54.378439  18.605984  arterial-urban   
90673      PL         day  50.253217  19.823732   smaller-rural   
24373      DE         day  50.931399   6.953686            city   
41597      SE         day  55.608149  13.003458            city   
40353      IT       night  41.918667  12.383545            city   

      road_condition            weather  solar_angle_elevation  month  hour  \
8459          normal             cloudy              36.560248      5    14   
63960         normal               ra

In [33]:
iou = data["iou"]
pq = data["pq"]
lrp = data["lrp"]

data = data.drop(columns=["iou", "pq", "lrp"])
data_indices = data.index.to_numpy()

In [34]:
all_columns = data.columns    
all_columns = all_columns.drop("sunshine_duration")

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9999
Columns: Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour', 'temperature_2m',
       'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover',
       'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m', 'weather_code',
       'laplacian', 'quality', 'brightness', 'noisiness', 'sharpness',
       'contrast', 'complexity', 'conf'],
      dtype='object')
Number of columns: 27


In [35]:
numeric_columns = ['lat', 'long', 'solar_angle_elevation', 'month', 'hour', 'temperature_2m',
       'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 
       'wind_speed_10m', 'conf',  'laplacian', 'quality', 'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity']
#'precipitation',

categorical_columns = ['time_of_day', 'country', 'road_type', 'road_condition',
       'weather', 'weather_code']

assert len(all_columns) == (len(categorical_columns) + len(numeric_columns)), "Columns not match"


# Assessor Tests

### Baseline

Random Prediction (Normal dist)

In [ ]:
random_iou = np.random.normal(loc=0.5, scale=0.5, size=len(data))
random_iou = np.clip(random_iou, 0, 1)
random_baseline_r2 = r2_score(iou, random_iou)
print(f"R2 score of random baseline: {random_baseline_r2:.4f}")
print(f"MAE of random baseline: {np.mean(np.abs(iou - random_iou)):.4f}")
print(f"MSE of random baseline: {np.mean((iou - random_iou)**2):.4f}")

R2 score of random baseline: -7.1478
MAE of random baseline: 0.3759
RMSE of random baseline: 0.4573


### Models

Split data into train 60% and validation 40%

In [37]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]


print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 5999 4000
y: 5999 4000
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour', 'temperature_2m',
       'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover',
       'cloud_cover_low', 'cloud_cover_mid', 'sunshine_duration',
       'wind_speed_10m', 'weather_code', 'laplacian', 'quality', 'brightness',
       'noisiness', 'sharpness', 'contrast', 'complexity', 'conf'],
      dtype='object')


Train models with cv and then test.

#### Linear Regression

In [39]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)],remainder='passthrough')

linear_reg = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])
cv_lr = cross_validate(linear_reg, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_lr["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_lr["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_lr["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_lr["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_lr["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_lr["test_neg_mean_squared_error"])}")

Mean CV train r2 score 0.7039821223174556
Mean CV test r2 score 0.6899361600389063
Mean CV train MAE score -0.060315713210989164
Mean CV test MAE score -0.061129751652124975
Mean CV train MSE score -0.007462823378944961
Mean CV test MSE score -0.007694220111973122


In [41]:
linear_reg.fit(X_train, y_train)

y_pred = linear_reg.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
lr_test_mae = np.mean(np.abs(y_test - y_pred))
lr_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lr_test_r2}")
print(f"MAE test score {lr_test_mae}")
print(f"MSE test score {lr_test_mse}")


R2 test score 0.6957116861348597
MAE test score 0.06154904269155297
MSE test score 0.008009368693633763


#### Decision Trees

In [42]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)],remainder='passthrough')

decision_tree = Pipeline([("preprocessor", preprocessor), ("model", DecisionTreeRegressor())])
cv_dt = cross_validate(decision_tree, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_dt["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_dt["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_dt["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_dt["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_dt["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_dt["test_neg_mean_squared_error"])}")

Mean CV train r2 score 1.0
Mean CV test r2 score 0.47338093030615835
Mean CV train MAE score 0.0
Mean CV test MAE score -0.07587428043025907
Mean CV train MSE score 0.0
Mean CV test MSE score -0.012975751707082482


In [43]:
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
lr_test_mae = np.mean(np.abs(y_test - y_pred))
lr_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lr_test_r2}")
print(f"MAE test score {lr_test_mae}")
print(f"MSE test score {lr_test_mse}")


R2 test score 0.48253876931541073
MAE test score 0.07528542070132323
MSE test score 0.013620430336510374


#### Random Forest

In [45]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)],remainder='passthrough')

rand_forest = Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor())])
cv_rf = cross_validate(rand_forest, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_rf["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_rf["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_rf["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_rf["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_rf["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_rf["test_neg_mean_squared_error"])}")

Mean CV train r2 score 0.9637655771516243
Mean CV test r2 score 0.7339675429120645
Mean CV train MAE score -0.019749469669708496
Mean CV test MAE score -0.053225247694138826
Mean CV train MSE score -0.000913556396790976
Mean CV test MSE score -0.006623043847383272


In [46]:
rand_forest.fit(X_train, y_train)
y_pred = rand_forest.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
lr_test_mae = np.mean(np.abs(y_test - y_pred))
lr_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lr_test_r2}")
print(f"MAE test score {lr_test_mae}")
print(f"MSE test score {lr_test_mse}")

R2 test score 0.7311201863850436
MAE test score 0.05491854307375104
MSE test score 0.007077358752831247


#### MLP

In [47]:
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_columns),('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)],remainder='passthrough')

mlp = Pipeline([("preprocessor", preprocessor), ("model", MLPRegressor())])
cv_mlp = cross_validate(mlp, X_train, y_train, cv=10, scoring=("r2", "neg_mean_absolute_error", "neg_mean_squared_error"), return_train_score=True)
print(f"Mean CV train r2 score {statistics.mean(cv_mlp["train_r2"])}")
print(f"Mean CV test r2 score {statistics.mean(cv_mlp["test_r2"])}")
print(f"Mean CV train MAE score {statistics.mean(cv_mlp["train_neg_mean_absolute_error"])}")
print(f"Mean CV test MAE score {statistics.mean(cv_mlp["test_neg_mean_absolute_error"])}")
print(f"Mean CV train MSE score {statistics.mean(cv_mlp["train_neg_mean_squared_error"])}")
print(f"Mean CV test MSE score {statistics.mean(cv_mlp["test_neg_mean_squared_error"])}")

Mean CV train r2 score 0.4673217469943964
Mean CV test r2 score 0.32763546963786544
Mean CV train MAE score -0.07913594842455954
Mean CV test MAE score -0.09180355884070991
Mean CV train MSE score -0.013395702188196098
Mean CV test MSE score -0.01686594166001022


In [48]:
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
lr_test_r2 = r2_score(y_test, y_pred)
lr_test_mae = np.mean(np.abs(y_test - y_pred))
lr_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lr_test_r2}")
print(f"MAE test score {lr_test_mae}")
print(f"MSE test score {lr_test_mse}")  

R2 test score 0.37819543624955576
MAE test score 0.10049764774224383
MSE test score 0.016366918411032518
